## Evolution over time vs mRS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from utils.utils import load_encrypted_xlsx
import os
os.environ["R_HOME"] = "/Library/Frameworks/R.framework/Versions/4.1/Resources"
from pymer4.models import Lmer
import statsmodels.api as sm


In [ ]:
registry_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx"
outcome_data_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx"
bp_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke.csv"
nor_annotated_bp_path = '/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke_nor_annotated.csv'
registry_pdms_correspondence_path = "/Users/jk1/Library/CloudStorage/OneDrive-UniversitédeGenève/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv"

output_dir = '/Users/jk1/temp/bp_dci/bp_trajectories'


In [ ]:
filter_noradenaline = True

In [ ]:
registry_df = load_encrypted_xlsx(registry_path)
if not filter_noradenaline:
    bp_df = pd.read_csv(bp_path, sep= ';', decimal='.')
else:
    nor_annotated_bp_df = pd.read_csv(nor_annotated_bp_path)
    bp_df = nor_annotated_bp_df[nor_annotated_bp_df['noradrenaline_concomitant'] == 0]
registry_pdms_correspondence_df = pd.read_csv(registry_pdms_correspondence_path)
outcome_df = load_encrypted_xlsx(outcome_data_path)

In [ ]:
# drop duplicates 
bp_df = bp_df.drop_duplicates(subset=['pNr', 
                                       'systole',
                                       'diastole',
                                       'mitteldruck',
                                       'timeBd'])

registry_df.drop_duplicates(inplace=True)
registry_df.dropna(subset=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'], inplace=True)

In [ ]:
bp_df = bp_df.merge(registry_pdms_correspondence_df, how="left", on="pNr")

In [ ]:
# <!-- fillna in mRS_FU_1y with mRS_2FU_2y and then mRS_3FU_5y -->
outcome_df['mRS_FU_1y'] = outcome_df['mRS_FU_1y'].fillna(outcome_df['mRS_2FU_2y'])
outcome_df['mRS_FU_1y'] = outcome_df['mRS_FU_1y'].fillna(outcome_df['mRS_3FU_5y'])

outcome_df['mRS_FU_1y'] = pd.to_numeric(outcome_df['mRS_FU_1y'], errors='coerce')
outcome_df['mRS_discharge'] = pd.to_numeric(outcome_df['mRS_discharge'], errors='coerce')

# if mRS_discharge == 6, set mRS_FU_1y to 6
outcome_df.loc[outcome_df['mRS_discharge'] == 6, 'mRS_FU_1y'] = 6

In [ ]:
outcome_df["mRS_FU_1y_int"] = pd.to_numeric(outcome_df["mRS_FU_1y"], errors="coerce")
bp_df["Date_birth"] = pd.to_datetime(bp_df["Date_birth"], format="%d.%m.%Y")
outcome_df["Date_birth"] = pd.to_datetime(outcome_df["Date_birth"])
registry_df['Fisher_Score'] = pd.to_numeric(registry_df['Fisher_Score'], errors='coerce')

In [ ]:
bp_df["mrs_1y"] = np.nan
for pnr in tqdm(bp_df["pNr"].unique()):
    sos_center_nr = bp_df[bp_df["pNr"] == pnr]["SOS-CENTER-YEAR-NO."].values[0]
    name = bp_df[bp_df["pNr"] == pnr]["JoinedName"].values[0]
    date_birth = bp_df[bp_df["pNr"] == pnr]["Date_birth"].values[0]
    mrs_values = outcome_df[(outcome_df["SOS-CENTER-YEAR-NO."] == sos_center_nr) &
                        (outcome_df["Name"] == name) &
                        (outcome_df["Date_birth"] == date_birth)]["mRS_FU_1y_int"]
    if len(mrs_values) == 0:
        mrs = np.nan
    else:
        mrs = mrs_values.values[0]

    bp_df.loc[bp_df["pNr"] == pnr, "mrs_1y"] = mrs

In [ ]:
registry_df = registry_df.drop_duplicates(subset=["SOS-CENTER-YEAR-NO.", "Date_birth", "Name"])
bp_df = bp_df.merge(registry_df[["SOS-CENTER-YEAR-NO.", "Date_birth", "Name", 'DCI_ischemia', 'Fisher_Score', 'WFNS'	]], how="left", left_on=["SOS-CENTER-YEAR-NO.", "Date_birth", "JoinedName"],
                right_on=["SOS-CENTER-YEAR-NO.", "Date_birth", "Name"])

In [ ]:
# dichotomize mrs_1y into 0-2 and 3-6
bp_df["mrs_1y_02"] = bp_df["mrs_1y"].isin([0, 1, 2]).astype(int)


In [ ]:
# get first measure (by timeBd) for every pNr
first_measure_df = bp_df.groupby("pNr").agg({"timeBd": "min"}).reset_index()
first_measure_df = first_measure_df.rename(columns={"timeBd": "first_timeBd"})
bp_df = bp_df.merge(first_measure_df, how="left", on="pNr")
bp_df["relative_timeBd"] = (pd.to_datetime(bp_df["timeBd"]) - pd.to_datetime(bp_df["first_timeBd"])).dt.total_seconds() / 3600
bp_df["relative_timeBd_days"] = bp_df["relative_timeBd"] / 24
bp_df["relative_timeBd_days_cat"] = bp_df["relative_timeBd_days"].apply(np.floor)
bp_df["relative_timeBd_hours_cat"] = bp_df["relative_timeBd"].apply(np.floor)

In [ ]:
# get median daily bp for each patient
daily_bp_df = bp_df.groupby(["pNr", "relative_timeBd_days_cat"]).agg({"systole": "median", "diastole": "median", "mitteldruck": "median", "mrs_1y": "first", "mrs_1y_02": "first", "DCI_ischemia": "first"}).reset_index()
daily_bp_df = daily_bp_df.rename(columns={"relative_timeBd_days_cat": "relative_timeBd_days"})
daily_bp_df = daily_bp_df.drop_duplicates(subset=["pNr", "relative_timeBd_days"])


In [ ]:
hourly_bp_df = bp_df.groupby(["pNr", "relative_timeBd_hours_cat"]).agg({"systole": "median", "diastole": "median", "mitteldruck": "median", "mrs_1y": "first", "mrs_1y_02": "first", "DCI_ischemia": "first"}).reset_index()
hourly_bp_df = hourly_bp_df.rename(columns={"relative_timeBd_hours_cat": "relative_timeBd_hours"})
hourly_bp_df = hourly_bp_df.drop_duplicates(subset=["pNr", "relative_timeBd_hours"])

In [ ]:
# 3 sublots
fig, axs = plt.subplots(3, 1, figsize=(10, 15))
# plot bp per mrs cat
sns.boxplot(data=daily_bp_df, x="relative_timeBd_days", y="systole", hue="mrs_1y_02", palette="Set2", showfliers=False, ax=axs[0])
axs[0].set_xlim(0, 21)

# tilt x axis labels
axs[0].set_xticklabels(axs[0].get_xticks(), rotation=45)
axs[0].set_xlabel("Relative time (days)")
axs[0].set_ylabel("Systolic BP (mmHg)")

sns.boxplot(data=daily_bp_df, x="relative_timeBd_days", y="diastole", hue="mrs_1y_02", palette="Set2", showfliers=False, ax=axs[1])
axs[1].set_xlim(0, 21)
# tilt x axis labels
axs[1].set_xticklabels(axs[1].get_xticks(), rotation=45)
axs[1].set_xlabel("Relative time (days)")
axs[1].set_ylabel("Diastolic BP (mmHg)")

sns.boxplot(data=daily_bp_df, x="relative_timeBd_days", y="mitteldruck", hue="mrs_1y_02", palette="Set2", showfliers=False, ax=axs[2])
axs[2].set_xlim(0, 21)
# tilt x axis labels
axs[2].set_xticklabels(axs[2].get_xticks(), rotation=45)
axs[2].set_xlabel("Relative time (days)")
axs[2].set_ylabel("Mean BP (mmHg)")
plt.tight_layout()

effect of BP on mrs

In [ ]:

temp_df = hourly_bp_df.copy()
# scale systole, diastole and mitteldruck
temp_df["systole"] = (temp_df["systole"] - temp_df["systole"].mean()) / temp_df["systole"].std()
temp_df["diastole"] = (temp_df["diastole"] - temp_df["diastole"].mean()) / temp_df["diastole"].std()
temp_df["mitteldruck"] = (temp_df["mitteldruck"] - temp_df["mitteldruck"].mean()) / temp_df["mitteldruck"].std()

map_model = Lmer("mrs_1y_02 ~ mitteldruck + (1|pNr)", data=temp_df, family="binomial")
map_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(map_model.summary())

sys_model = Lmer("mrs_1y_02 ~ systole + (1|pNr)", data=temp_df, family="binomial")
sys_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(sys_model.summary())

dia_model = Lmer("mrs_1y_02 ~ diastole + (1|pNr)", data=temp_df, family="binomial")
dia_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(dia_model.summary())

print("--------------------")
print(f'MAP on mrs, pval: {map_model.coefs["P-val"].to_dict()["mitteldruck"]:.3f}, coef: {map_model.coefs["Estimate"].to_dict()["mitteldruck"]:.3f}')
print(f'Systolic on mrs, pval: {sys_model.coefs["P-val"].to_dict()["systole"]:.3f}, coef: {sys_model.coefs["Estimate"].to_dict()["systole"]:.3f}')
print(f'Diastolic on mrs, pval: {dia_model.coefs["P-val"].to_dict()["diastole"]:.3f}, coef: {dia_model.coefs["Estimate"].to_dict()["diastole"]:.3f}')


BP by mrs category

In [ ]:
# linear regression of sytole, diastole and mitteldruck by mrs_1y_02

temp_df = hourly_bp_df.copy()
# scale systole, diastole and mitteldruck
temp_df["systole"] = (temp_df["systole"] - temp_df["systole"].mean()) / temp_df["systole"].std()
temp_df["diastole"] = (temp_df["diastole"] - temp_df["diastole"].mean()) / temp_df["diastole"].std()
temp_df["mitteldruck"] = (temp_df["mitteldruck"] - temp_df["mitteldruck"].mean()) / temp_df["mitteldruck"].std()
temp_df = temp_df.dropna(subset=["systole", "diastole", "mitteldruck", "mrs_1y"])

map_model = Lmer("mitteldruck ~ mrs_1y + (1|pNr)", data=temp_df, family="gaussian")
map_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(map_model.summary())

sys_model = Lmer("systole ~ mrs_1y + (1|pNr)", data=temp_df, family="gaussian")
sys_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(sys_model.summary())

dia_model = Lmer("diastole ~ mrs_1y + (1|pNr)", data=temp_df, family="gaussian")
dia_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(dia_model.summary())

print("--------------------")
print(f'MAP per mrs, pval: {map_model.coefs["P-val"].to_dict()["mrs_1y"]:.3f}, with estimate: {map_model.coefs["Estimate"].to_dict()["mrs_1y"]:.3f}')
print(f'Systolic per mrs, pval: {sys_model.coefs["P-val"].to_dict()["mrs_1y"]:.3f}, with estimate: {sys_model.coefs["Estimate"].to_dict()["mrs_1y"]:.3f}')
print(f'Diastolic per mrs, pval: {dia_model.coefs["P-val"].to_dict()["mrs_1y"]:.3f}, with estimate: {dia_model.coefs["Estimate"].to_dict()["mrs_1y"]:.3f}')

In [ ]:
# save summary of models to csv for map, sys and dia
summary_df = pd.DataFrame({
    "model": ["MAP", "Systolic", "Diastolic"],
    "pval": [map_model.coefs["P-val"].to_dict()["mrs_1y"], sys_model.coefs["P-val"].to_dict()["mrs_1y"], dia_model.coefs["P-val"].to_dict()["mrs_1y"]],
    "estimate": [map_model.coefs["Estimate"].to_dict()["mrs_1y"], sys_model.coefs["Estimate"].to_dict()["mrs_1y"], dia_model.coefs["Estimate"].to_dict()["mrs_1y"]],
    "formula": [map_model.formula, sys_model.formula, dia_model.formula]
})

# file name depends on nor filter
if filter_noradenaline:
    filename = "bp_mrs_models_summary_filtered_nor.csv"
else:
    filename = "bp_mrs_models_summary_unfiltered_nor.csv"
summary_df.to_csv(os.path.join(output_dir, filename), index=False)

among patients without DCI

In [ ]:
no_dci_df = daily_bp_df[daily_bp_df["DCI_ischemia"] == 0]


# 3 sublots
fig, axs = plt.subplots(3, 1, figsize=(10, 15))
# plot bp per mrs cat
sns.boxplot(data=no_dci_df, x="relative_timeBd_days", y="systole", hue="mrs_1y_02", palette="Set2", showfliers=False, ax=axs[0])
axs[0].set_xlim(0, 21)

# tilt x axis labels
axs[0].set_xticklabels(axs[0].get_xticks(), rotation=45)
axs[0].set_xlabel("Relative time (days)")
axs[0].set_ylabel("Systolic BP (mmHg)")

sns.boxplot(data=no_dci_df, x="relative_timeBd_days", y="diastole", hue="mrs_1y_02", palette="Set2", showfliers=False, ax=axs[1])
axs[1].set_xlim(0, 21)
# tilt x axis labels
axs[1].set_xticklabels(axs[1].get_xticks(), rotation=45)
axs[1].set_xlabel("Relative time (days)")
axs[1].set_ylabel("Diastolic BP (mmHg)")

sns.boxplot(data=no_dci_df, x="relative_timeBd_days", y="mitteldruck", hue="mrs_1y_02", palette="Set2", showfliers=False, ax=axs[2])
axs[2].set_xlim(0, 21)
# tilt x axis labels
axs[2].set_xticklabels(axs[2].get_xticks(), rotation=45)
axs[2].set_xlabel("Relative time (days)")
axs[2].set_ylabel("Mean BP (mmHg)")

plt.title("BP in patients without DCI")

plt.tight_layout()


### subdivide in days 0-4 / 4-14

- Ref: SAH guidelines

In [ ]:
daily_bp_df.head()

In [ ]:
daily_bp_d4_14_df = daily_bp_df[(daily_bp_df["relative_timeBd_days"] >= 4) & (daily_bp_df["relative_timeBd_days"] <= 14)]
daily_bp_d4_14_df = daily_bp_d4_14_df.dropna(subset=["systole", "diastole", "mitteldruck", "mrs_1y"])
daily_bp_d4_14_df = daily_bp_d4_14_df.drop_duplicates(subset=["pNr", "relative_timeBd_days"])

hourly_bp_d4_14_df = hourly_bp_df[(hourly_bp_df["relative_timeBd_hours"] >= 96) & (hourly_bp_df["relative_timeBd_hours"] <= 336)]
hourly_bp_d4_14_df = hourly_bp_d4_14_df.dropna(subset=["systole", "diastole", "mitteldruck", "mrs_1y"])
hourly_bp_d4_14_df = hourly_bp_d4_14_df.drop_duplicates(subset=["pNr", "relative_timeBd_hours"])

bp_d4_14_df = bp_df[(bp_df["relative_timeBd_days"] >= 4) & (bp_df["relative_timeBd_days"] <= 14)]
bp_d4_14_df = bp_d4_14_df.dropna(subset=["systole", "diastole", "mitteldruck", "mrs_1y"])
bp_d4_14_df = bp_d4_14_df.drop_duplicates(subset=["pNr", "relative_timeBd"])

In [ ]:
temp_df = hourly_bp_d4_14_df.copy()

map_model = Lmer("mitteldruck ~ mrs_1y + (1|pNr)", data=temp_df, family="gaussian")
map_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(map_model.summary())

sys_model = Lmer("systole ~ mrs_1y + (1|pNr)", data=temp_df, family="gaussian")
sys_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(sys_model.summary())

dia_model = Lmer("diastole ~ mrs_1y + (1|pNr)", data=temp_df, family="gaussian")
dia_model.fit(control='optimizer="bobyqa", optCtrl=list(maxfun=100000)')
# print(dia_model.summary())

print("--------------------")
print(f'MAP per mrs, pval: {map_model.coefs["P-val"].to_dict()["mrs_1y"]:.3f}, with estimate: {map_model.coefs["Estimate"].to_dict()["mrs_1y"]:.3f}')
print(f'Systolic per mrs, pval: {sys_model.coefs["P-val"].to_dict()["mrs_1y"]:.3f}, with estimate: {sys_model.coefs["Estimate"].to_dict()["mrs_1y"]:.3f}')
print(f'Diastolic per mrs, pval: {dia_model.coefs["P-val"].to_dict()["mrs_1y"]:.3f}, with estimate: {dia_model.coefs["Estimate"].to_dict()["mrs_1y"]:.3f}')